#Load the bronze data

In [0]:
df = spark.table('aws_s3_catalog.bronze.products')
display(df.limit(10))

#process the data for silver layer

In [0]:
from pyspark.sql import functions as F

In [0]:
df.select('category').distinct().show()

###Trim and collapse whitespace

In [0]:
df = df.select(
    "product_name", F.length("product_name").alias("product_name_len"),
    "product_id",   F.length("product_id").alias("product_id_len"),
    "category",     F.length("category").alias("category_len"),
)
display(df.limit(10))

In [0]:
df = (
    df.withColumn('product_name',F.trim(F.regexp_replace('product_name', r'\s+', ' ')))
    .withColumn('product_id', F.trim('product_id'))
    .withColumn('category', F.trim('category'))
)

display(df.limit(10))

In [0]:
df = df.select(
    "product_name", F.length("product_name").alias("product_name_len"),
    "product_id",   F.length("product_id").alias("product_id_len"),
    "category",     F.length("category").alias("category_len"),
)
display(df.limit(10))

###Fix the 'protien' typo

In [0]:
df = (
    df
    .withColumn("product_name", F.regexp_replace("product_name", "(?i)protien", "Protein"))
    .withColumn("category", F.regexp_replace("category", "protien", "protein"))
)
display(df)

###Remove rows with invalid IDs or names

In [0]:
df = (
    df
    .filter(F.col("product_id").rlike(r"^\d+$"))
    .filter(F.col("product_name").isNotNull() & (F.col("product_name") != ""))
)
display(df)

###Cast product_id to a number

In [0]:
df = df.withColumn("product_id", F.col("product_id").cast("bigint"))

###Remove len columns

In [0]:
df.columns

In [0]:
df = df.drop("product_name_len", "product_id_len", "category_len")

###Remove duplicates

In [0]:
df = df.dropDuplicates(["product_id"])
display(df)

###add an adult timestamp

In [0]:
df = df.withColumn("ingested_at", F.current_timestamp())
display(df)

###Load table to silver layer

In [0]:

df.printSchema()
display(df)

In [0]:
df.write.mode("overwrite").option("overwriteSchema", "true") \
  .saveAsTable("aws_s3_catalog.silver.products")